# 02 — Finite-Temperature Phase Diagram

Map out the crossover from power-law (Luther-Emery) to exponential (gapped) behavior
as a function of temperature $T = 1/\beta$ and hopping exponent $\alpha$.

**Goal**: identify the high-$T$ onset of superconducting correlations as a function of $\alpha$.

**Plan** (fill in as data becomes available)
1. β sweep at fixed $(n_\sigma = 0.4, \alpha = 0.8, U = -5)$ — track $\eta_P$ vs $T$
2. Repeat for each $\alpha \in \{0.5, 0.8, 1.0, 1.5, 2.0\}$
3. Plot: $\eta_P(T, \alpha)$ — build the $(T, \alpha)$ phase diagram

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
import sys, os

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))
from analysis.extract import SimParams, extract_equal_time

plt.rcParams.update({'figure.dpi': 120, 'font.size': 12})

DATA_BASE = "/nfs/home/gissa/Hubbard"  # adjust as needed


def power_law_fit(r, P, r_min=5, r_max=None):
    if r_max is None:
        r_max = r.max()
    mask = (r >= r_min) & (r <= r_max) & (P > 0)
    slope, intercept, rval, _, stderr = linregress(np.log(r[mask]), np.log(P[mask]))
    return -slope, np.exp(intercept), rval**2

## Step 4a: β sweep at $\alpha = 0.8$, $R_{\max} = 50$

In [ ]:
# Parameters (update mu_lr_star from 00_sanity.ipynb after LR density calibration)
U         = -5.0
L         = 100
alpha     = 0.8
Rmax      = 50
mu_lr_star = -2.4   # <-- update

betas = [20.0, 30.0, 40.0, 60.0, 80.0, 100.0]

eta_vs_beta = {}  # beta -> eta_P

for beta in betas:
    params = SimParams(U=U, mu=mu_lr_star, beta=beta, L=L, alpha=alpha, Rmax=Rmax)
    try:
        data = extract_equal_time(params, 'pair', 'position', base=DATA_BASE)
    except FileNotFoundError as e:
        print(f"[skip] β={beta}: {e}")
        continue
    r, P, err = data['r'], data['values'], data['errors']
    eta, _, r2 = power_law_fit(r, P)
    eta_vs_beta[beta] = eta
    print(f"β={beta:.0f}  η_P={eta:.4f}  R²={r2:.3f}")

print()
print('η_P vs β:', eta_vs_beta)

## Step 4b: Phase diagram — $\eta_P(T)$ for multiple $\alpha$

Placeholder: repeat the β sweep above for each α and collect into `eta_table`.

In [ ]:
# eta_table: dict of {alpha: {beta: eta_P}}
# Fill this in as data becomes available.
eta_table = {
    # 0.5: {20: ..., 40: ..., ...},
    # 0.8: eta_vs_beta,
    # 1.0: {...},
}

if len(eta_vs_beta) > 1:
    eta_table[alpha] = eta_vs_beta

if eta_table:
    fig, ax = plt.subplots(figsize=(7, 5))
    for a, d in sorted(eta_table.items()):
        betas_sorted = sorted(d.keys())
        Ts = [1.0 / b for b in betas_sorted]
        etas = [d[b] for b in betas_sorted]
        ax.plot(Ts, etas, 'o-', label=f'α={a}')

    ax.set_xlabel(r'$T = 1/\beta$')
    ax.set_ylabel(r'$\eta_P$')
    ax.set_title(r'Finite-$T$ phase diagram: $\eta_P(T, \alpha)$')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../results/phase_diagram.pdf')
    plt.show()
else:
    print("No data yet. Run simulations first.")